In [ ]:
# 1. Cài đặt thư viện thiếu
!pip install gensim

import os
import glob
import pandas as pd
import numpy as np
import tensorflow as tf
from google.colab import drive
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report
from gensim.models import Word2Vec

# 2. Kết nối Drive và Giải nén
drive.mount('/content/drive')
!tar -xzvf "/content/drive/MyDrive/hwu.tar.gz"

# 3. Hàm đọc dữ liệu thông minh (Tự động nhận diện dấu Tab/Phẩy và sửa lỗi 1 class)
def load_hwu_data(pattern):
    path = glob.glob(f"**/*{pattern}*.csv", recursive=True)[0]
    # sep=None giúp pandas tự đoán dấu phân tách (\t hoặc ,)
    df = pd.read_csv(path, sep=None, engine='python', header=None)
    if df.shape[1] >= 2:
        df = df[[0, 1]]
        df.columns = ['text', 'intent']
    return df

df_train = load_hwu_data('train')
df_val = load_hwu_data('val')
df_test = load_hwu_data('test')

# 4. Mã hóa nhãn
le = LabelEncoder()
y_train = le.fit_transform(df_train['intent'])
y_val = le.transform(df_val['intent'])
y_test = le.transform(df_test['intent'])
num_classes = len(le.classes_)

print(f"--- Dữ liệu đã sẵn sàng ---")
print(f"Số lượng nhãn: {num_classes}")
print(f"Mẫu dữ liệu: \n{df_train.head(2)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 76.5 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
hwu/
hwu/categories.json
hwu/train_5.csv
hwu/train_10.csv
hwu/val.csv
hwu/test.csv
hwu/train.csv
--- Dữ liệu đã sẵn sàng ---
Số lượng nhãn: 65
Mẫu dữ liệu: 
                              text       intent
0                             text     category
1  remind me about my alarms today  alarm_query


In [ ]:
# --- Nhiệm vụ 1: TF-IDF + Logistic Regression ---
model_tfidf = make_pipeline(TfidfVectorizer(max_features=5000), LogisticRegression(max_iter=1000))
model_tfidf.fit(df_train['text'], y_train)
acc_tfidf = accuracy_score(y_test, model_tfidf.predict(df_test['text']))
print(f"Nhiệm vụ 1 - TF-IDF Accuracy: {acc_tfidf:.4f}")

# --- Nhiệm vụ 2: Word2Vec Average + Dense Layer ---
sentences = [text.split() for text in df_train['text']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

def get_avg_v(text):
    vecs = [w2v_model.wv[w] for w in text.split() if w in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

X_train_w2v = np.array([get_avg_v(t) for t in df_train['text']])
X_test_w2v = np.array([get_avg_v(t) for t in df_test['text']])

model_dense = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(100,)),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])
model_dense.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_dense.fit(X_train_w2v, y_train, epochs=15, batch_size=32, verbose=0)
acc_w2v_avg = model_dense.evaluate(X_test_w2v, y_test, verbose=0)[1]
print(f"Nhiệm vụ 2 - W2V Avg Accuracy: {acc_w2v_avg:.4f}")

Nhiệm vụ 1 - TF-IDF Accuracy: 0.5738


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Nhiệm vụ 2 - W2V Avg Accuracy: 0.0167


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tiền xử lý chuỗi
max_len = 25
tokenizer = Tokenizer(num_words=10000, oov_token="<UNK>")
tokenizer.fit_on_texts(df_train['text'])
vocab_size = len(tokenizer.word_index) + 1

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(df_train['text']), maxlen=max_len)
X_val_pad = pad_sequences(tokenizer.texts_to_sequences(df_val['text']), maxlen=max_len)
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(df_test['text']), maxlen=max_len)

# --- Nhiệm vụ 3: LSTM với Pre-trained Embedding (Word2Vec) ---
emb_matrix = np.zeros((vocab_size, 100))
for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        emb_matrix[i] = w2v_model.wv[word]

model_lstm_pre = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 100, weights=[emb_matrix], input_length=max_len, trainable=False),
    tf.keras.layers.LSTM(128, dropout=0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# --- Nhiệm vụ 4: LSTM học Embedding từ đầu (Scratch) ---
model_lstm_scratch = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 100, input_length=max_len),
    tf.keras.layers.LSTM(128, dropout=0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# Hàm huấn luyện chung
def train_and_eval(model, name):
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    print(f"\n--- Đang huấn luyện mô hình: {name} ---")
    model.fit(X_train_pad, y_train, validation_data=(X_val_pad, y_val),
              epochs=20, batch_size=64, callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)], verbose=1)
    acc = model.evaluate(X_test_pad, y_test, verbose=0)[1]
    return acc

acc_pre = train_and_eval(model_lstm_pre, "LSTM Pre-trained")
acc_scratch = train_and_eval(model_lstm_scratch, "LSTM Scratch")

print("\n" + "="*30)
print(f"KẾT QUẢ CUỐI CÙNG:")
print(f"1. TF-IDF Accuracy: {acc_tfidf:.4f}")
print(f"2. W2V Avg Accuracy: {acc_w2v_avg:.4f}")
print(f"3. LSTM Pre-trained: {acc_pre:.4f}")
print(f"4. LSTM Scratch: {acc_scratch:.4f}")
print("="*30)


--- Đang huấn luyện mô hình: LSTM Pre-trained ---
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 243ms/step - accuracy: 0.0000e+00 - loss: 4.1750 - val_accuracy: 0.0306 - val_loss: 4.1738
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 102ms/step - accuracy: 0.0172 - loss: 4.1738 - val_accuracy: 0.0074 - val_loss: 4.1738
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 257ms/step - accuracy: 0.0173 - loss: 4.1732 - val_accuracy: 0.0074 - val_loss: 4.1737
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 306ms/step - accuracy: 0.0154 - loss: 4.1726 - val_accuracy: 0.0074 - val_loss: 4.1734
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step - accuracy: 0.0173 - loss: 4.1712 - val_accuracy: 0.0074 - val_loss: 4.1731
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - accuracy: 0.0113 - loss: 4.1710 - val_accuracy: 0.0074 - val_loss: 4.1732
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 106ms/step - accuracy: 0.0180 - loss: 4.1711 - val_accuracy: 0.0074 - val_loss: 4.1731
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step - accuracy: 0.0078 - loss: 4.1722 - val_accuracy: 0.0074 - val_loss: 4.1729

In [ ]:
def deep_analyze(text):
    # Tiền xử lý cho từng loại model
    seq = pad_sequences(tokenizer.texts_to_sequences([text]), maxlen=max_len)
    avg_v = np.array([get_avg_v(text)])

    # Dự đoán
    p_tfidf = le.inverse_transform(model_tfidf.predict([text]))[0]
    p_scratch = le.inverse_transform([np.argmax(model_lstm_scratch.predict(seq, verbose=0))])[0]

    print(f"\nCâu test: '{text}'")
    print(f"  > TF-IDF dự đoán: {p_tfidf}")
    print(f"  > LSTM (Scratch) dự đoán: {p_scratch}")

print("\n--- PHÂN TÍCH ĐỊNH TÍNH ---")
hard_cases = [
    "can you remind me to not call my mom",
    "is it going to be sunny or rainy tomorrow",
    "find a flight from new york to london but not through paris"
]

for case in hard_cases:
    deep_analyze(case)


--- PHÂN TÍCH ĐỊNH TÍNH ---

Câu test: 'can you remind me to not call my mom'
  > TF-IDF dự đoán: calendar_set
  > LSTM (Scratch) dự đoán: general_commandstop

Câu test: 'is it going to be sunny or rainy tomorrow'
  > TF-IDF dự đoán: general_negate
  > LSTM (Scratch) dự đoán: alarm_set

Câu test: 'find a flight from new york to london but not through paris'
  > TF-IDF dự đoán: general_negate
  > LSTM (Scratch) dự đoán: datetime_convert


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

# 1. Lấy dự đoán từ tất cả các pipeline
y_pred_tfidf = model_tfidf.predict(df_test['text'])
y_pred_w2v = np.argmax(model_dense.predict(X_test_w2v, verbose=0), axis=1)
y_pred_pre = np.argmax(model_lstm_pre.predict(X_test_pad, verbose=0), axis=1)
y_pred_scratch = np.argmax(model_lstm_scratch.predict(X_test_pad, verbose=0), axis=1)

# 2. Tính toán Loss (chỉ dành cho các mô hình Keras)
loss_w2v = model_dense.evaluate(X_test_w2v, y_test, verbose=0)[0]
loss_pre = model_lstm_pre.evaluate(X_test_pad, y_test, verbose=0)[0]
loss_scratch = model_lstm_scratch.evaluate(X_test_pad, y_test, verbose=0)[0]

# 3. Tổng hợp kết quả
results = [
    ["TF-IDF + Logistic Regression", f1_score(y_test, y_pred_tfidf, average='macro'), "N/A"],
    ["Word2Vec (Avg) + Dense", f1_score(y_test, y_pred_w2v, average='macro'), loss_w2v],
    ["Embedding (Pre-trained) + LSTM", f1_score(y_test, y_pred_pre, average='macro'), loss_pre],
    ["Embedding (Scratch) + LSTM", f1_score(y_test, y_pred_scratch, average='macro'), loss_scratch]
]

print(f"\n{'Pipeline':<35} | {'F1-score (Macro)':<20} | {'Test Loss':<10}")
print("-" * 75)
for res in results:
    f1_val = f"{res[1]:.4f}"
    loss_val = f"{res[2]:.4f}" if isinstance(res[2], float) else res[2]
    print(f"{res[0]:<35} | {f1_val:<20} | {loss_val}")


Pipeline                            | F1-score (Macro)     | Test Loss 
---------------------------------------------------------------------------
TF-IDF + Logistic Regression        | 0.5386               | N/A
Word2Vec (Avg) + Dense              | 0.0033               | 4.1676
Embedding (Pre-trained) + LSTM      | 0.0002               | 4.1702
Embedding (Scratch) + LSTM          | 0.1110               | 3.6237
